#### Load the Env Variables

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
import os
apiKey = os.getenv("GROQ_API_KEY")

In [ ]:
apiKey

#### Simple QnA Bot without LangChain - With Groq

In [ ]:
from groq import Groq

client = Groq()
completion = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
      {
        "role": "user",
        "content": "Explain me the Gen-ai"
      }
    ],
    max_completion_tokens=100
)

In [ ]:
print(completion.choices[0].message.content)

#### Simple QnA Bot with LangChain - With Groq

In [ ]:
from langchain_groq import ChatGroq

In [ ]:
llm = ChatGroq(model="openai/gpt-oss-120b")
ans = llm.invoke("Hello, Hows You ?")

In [ ]:
ans.content

##### Chat with OpenAI

In [ ]:
from langchain_openai import ChatOpenAI

In [ ]:
openaiLlm = ChatOpenAI(model="gpt-5.1", api_key=os.getenv("OPENAI_API_KEY"))

In [ ]:
res = openaiLlm.invoke("Who is pm of india ?")

In [ ]:
res.content

#### Common LLM Calls with - `init_chat_model`

In [ ]:
from langchain.chat_models import init_chat_model

In [ ]:

llm = init_chat_model(model=os.getenv("MODEL", "openai/gpt-oss-20b"), 
                      model_provider=os.getenv("PROVIDER", "groq"))


In [ ]:
res = llm.invoke("How to write a second max finder code")
# res

In [ ]:
print(res)

#### Core Components of `LangChain`
##### 1. `Messages`

#### Promte - LLM -> Three types / role of prompt ['user', 'ai', 'system' ]

In [ ]:

llm = init_chat_model(model=os.getenv("MODEL", "openai/gpt-oss-20b"), 
                      model_provider=os.getenv("PROVIDER", "groq"))

res = llm.invoke([ 
                  { "role":"system", "content":"You are a 7 years of python developer, but you have no idea about java programming language, if user ask for java, then just reply that 'I don't know java.'" }, 
                  { "role":"user", "content":"How to find second largest number in array using Java" } 
                  ])


In [ ]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage
prompts = [
    SystemMessage(content="You are a 7 years of python developer, but you have no idea about java programming language, if user ask for java, then just reply that 'I don't know java.'"),
    HumanMessage(content="How to find second largest number in array using Java")
]

res = llm.invoke(prompts)

##### 2. `Prompt Templates`

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    {"role":"system", "content":"You are a 7 years of {language} developer"}, 
    {"role":"user", "content":"{query}"}, 
])
prompt.invoke({})

#### 3. LCEL - `|` Operator

In [ ]:
question = "How to map the array"
language = "javascript"

finalprompt = prompt.invoke({"language":language, "query":question})

res = llm.invoke(finalprompt)
# print(res.content)

##### `LangChain` Chain

In [ ]:
from langchain.messages import AIMessage

def getContent(llmResponse:AIMessage):
    return llmResponse.content


def shortTheMessage(msg:str):
    return msg[:100]

In [ ]:
qna_chain = prompt | llm | getContent | shortTheMessage

In [ ]:
res = qna_chain.invoke({"language":"english", "query":"what is gen-ai"})

In [ ]:
print(len(res)) 
print(res)

##### 4. `Memory` - Conver. History

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-20b")

In [ ]:
res = llm.invoke("Who is Pm of india ?")

In [ ]:
res.content

In [ ]:
res = llm.invoke("What is his age ?")

In [ ]:
res.content

#### Implement history...

In [ ]:
history = []

In [ ]:
while True:
    query = input("User: ")
    if query == "exit":
        break
    
    history.append(HumanMessage(query))
    res = llm.invoke(history)
    history.append(AIMessage(content=res.content))
    print("AI: ", res.content)

In [ ]:
history

##### 5. `Structured output`

In [ ]:
from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-20b")

In [ ]:
res = llm.invoke("Hello, My name is Sarthak Jain, age is 22 and email is sarthakjain4452@gmail.com, Can you please parse the name, email, and age in the form of json.")

In [ ]:
print(res.content)

### Solution using pydantic classes

In [ ]:


from pydantic import BaseModel, Field

class User(BaseModel):
    name:str = Field(description="User full name")
    age:int = Field(description="User age in years")
    email:str = Field(description="User email address")


In [ ]:
llm_with_struct_out = llm.with_structured_output(User)

In [ ]:
res = llm_with_struct_out.invoke("Hello, My name is Sarthak Jain, age is 22 and email is sarthakjain4452@gmail.com, Can you please parse the name, email, and age in the form of json.")

In [ ]:
res.model_dump()

In [ ]:
res.name

#### 6. `Streaming`

In [ ]:
from langchain_groq import ChatGroq
from time import sleep
llm = ChatGroq(model="openai/gpt-oss-20b", streaming=True)

In [ ]:
res = llm.stream("Explain me Machine learning in 200 words")

In [ ]:
for chunk in res:
    print(chunk.content, end='', flush=True)
    sleep(0.005)

#### 7. `AI Agents`
7.1 Defining `tools`

In [ ]:
from langchain.tools import tool

@tool
def send_email(email:str):
    """ I can send email to any email address  """
    print("[TOOL] Called", email)
    return "Email Sent Successfully"

@tool
def add_numbers(a:int, b:int):
    """Add two number and return the sum"""
    return a + b

@tool
def multiply_numbers(a:int, b:int):
    """Multiply two number and return the product"""
    return a * b

In [ ]:
send_email.invoke({"email":"email@gmail.com"})

7.2 Agent -  `create_agent`

In [ ]:
from langchain.agents import create_agent

In [ ]:
agent = create_agent(
    system_prompt="""
        You always call tools for user question. 
    """,
    model=llm,
    tools=[send_email, add_numbers, multiply_numbers]
)

In [ ]:
agent

In [ ]:
que = "What is (2 + 3) * 7"
res = agent.invoke({ "messages": [ {"role":"user", "content":que} ] })

In [ ]:
for msg in res["messages"]:
    msg.pretty_print()

#### 7.3 Agent -  `memory`

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

In [ ]:
agent = create_agent(
    system_prompt="""
        You always call tools for user question. 
    """,
    model=llm,
    tools=[send_email, add_numbers, multiply_numbers],
    checkpointer=InMemorySaver()
)

In [ ]:
while True:
    query = input("User: ")
    if query == "exit":
        break
    
    res = agent.invoke(
        { "messages": [ {"role":"user", "content":query} ] },
        {"configurable": {"thread_id": "user_1"}}
        )
    
    ans = res["messages"][-1].content
    print("AI: ", ans)

#### 8. `Prebuilt middleware`

###### `Summarization` middleware

In [ ]:
from langchain.agents.middleware import SummarizationMiddleware

CHAT = [
    "My order id is ORD-4471.",
    "It was a Nikon Z50 camera.",
    "The box came open.",
    "The lens cap is missing.",
    "I want a replacement, not a refund.",
    'I love react js, and want to learn it',
    'Who is PM of India ?'
]

agent = create_agent(
    system_prompt="""
        You are a helpfull assistant, reply in one short line
    """,
    model=llm,
    tools=[],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("messages", 6),
            keep=("messages", 2)
        )
    ]
)


In [ ]:
config = {"configurable": {"thread_id": "user_2"}}

for msg in CHAT:
    agent.invoke(
        { "messages": [ {"role":"user", "content":msg} ] },
        config=config
    )
    
    print("Messages in history: ", len(agent.get_state(config).values["messages"]))
    

In [ ]:
# agent.get_state(config).values["messages"]

In [ ]:
res = agent.invoke(
        { "messages": [ {"role":"user", "content":"What is my camera name"} ] },
        config=config
    )

In [ ]:
print(res["messages"][-1].content)

##### `PIIMiddleware` 

In [ ]:
from langchain.agents.middleware import PIIMiddleware

agent = create_agent(
    model=llm,
    tools=[],
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="redact", apply_to_input=True),
    ],
)

In [ ]:
query = "Hello, I received email from sarthak@gmail.com for the payment of credit card 4111 1111 1111 1111"
res = agent.invoke({
    "messages": [ {"role":"user", "content":query} ]
})


In [ ]:
res["messages"][0].content